# Contoso AI Gaming Companion: growth analysis

**Current scope:** audited preparation, four mandatory answers, feature-usage
and error associations, separate retention/conversion regressions, player groups,
and conversion journeys. Definitions were confirmed with the user before fitting.
Source-ledger and independent numerical checks support the results.

Source: [interview case](https://github.com/vejadu/InterviewCase-Gaming-PC-App).
This is public **synthetic example data**, not real customer evidence.
The downloaded statement and pinned commit are in `source/`.


## Problem and decision

The product team wants to identify growth and product improvement opportunities
for a PC app with an AI Game Optimizer, AI Game Coach (Chat), and AI Game Agent.
Mandatory questions cover new Free/Paid accounts, daily and weekly retention,
Free-to-Paid conversion, and Paid cancellations. Optional work examines feature
usage, errors, segments, and conversion timing.

**Population and window, as documented by the source (not yet validated):**
accounts starting March 3–7, 2025, followed for days 0–59; the statement describes
300 players and a last possible observation of May 5, 2025.
The grain is an observed active player-day; the independent unit is the player.
The source describes 60 players per start date.

The eventual assignment asks for a 10–15 minute leadership presentation,
a 10–15 minute technical discussion, and reproducible code. These are presentation
lengths, not an assumed analysis time budget. The notebook and checked evidence
also support the separately requested business and technical presentation.


## Definitions to preserve

- A missing player-day means no observed activity; it does not prove cancellation.
- Daily retention means active on a specific tenure day; weekly retention means
  active at least once in a defined tenure-week window. Use the day-0 cohort denominator.
- Plan and billing status are recorded at day end. A Free-to-Paid change appears
  on the upgrade day; recorded cancellation appears on the final Paid row.
  Day-0 plan is therefore an observed end-of-day starting classification;
  same-day signup/upgrade ordering cannot be recovered from daily snapshots.
- Feature totals include successes and errors. Daily limits are Optimizer/Coach/Agent:
  Free 100/10/5 and Paid 200/50/20. Limits differ, so raw usage comparisons need context.
- Usage and error relationships with outcomes will be descriptive associations.
  This synthetic, observational dataset does not identify causal product effects.


## Packages and local paths

Use **Python 3 (Contoso gaming case)**. These imports use existing project packages;
plotting packages are loaded for later use, but no charts are created here.
The path lookup supports launching from the project root or this case folder.


## How to read this notebook

Short cells state the question, show important definitions/formulas and display
the results. Repeated transformations, audit bookkeeping, model fitting, plots
and independent checks live in [case_helpers/](case_helpers/). Function calls
use explicit named inputs; helper docstrings describe their grain and assumptions.

Two small classes collect the run's records: `AuditTrail` owns quality/audit
ledgers and `RegressionCollection` owns fitted models and diagnostics. Data
calculations remain ordinary functions. [CODE_GUIDE.md](CODE_GUIDE.md) maps every
stage to its implementation. The four mandatory questions retain separate cells.


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import platform
import sys

import numpy as np
import pandas as pd
from IPython.display import display

project_root = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "submission.json").is_file()
)
case_folder = project_root
data_path = project_root / "data" / "contoso_gaming_pc_app_output.csv"

In [ ]:
environment = {
    "interpreter": sys.executable,
    "Python": platform.python_version(),
}
for package in ["pandas", "numpy", "matplotlib", "seaborn", "nbformat", "nbclient"]:
    environment[package] = importlib.metadata.version(package)
display(pd.Series(environment, name="Installed version or path"))

# Helpers contain the audited transformations and checks; see CODE_GUIDE.md.
sys.path.insert(0, str(case_folder))
from case_helpers import audit as audit_helpers
from case_helpers import preparation, validation, metrics, features, modeling, plotting, reporting
from case_helpers.audit import AuditTrail
from case_helpers.modeling import RegressionCollection
from collections import Counter, defaultdict
from IPython.display import Markdown

## Load the supplied data

Read the CSV without parsing dates, filtering, renaming, filling missing values,
or changing records. Pandas infers in-memory types; the source bytes stay unchanged.
The SHA-256 fingerprint records the exact input used. This is loading verification,
not a data-quality audit.


In [ ]:
source_sha256 = hashlib.sha256(data_path.read_bytes()).hexdigest()
activity_raw = pd.read_csv(data_path)

print(f"Source: {data_path.relative_to(project_root)}")
print(f"SHA-256: {source_sha256}")
print(f"Loaded table: {activity_raw.shape[0]:,} rows × {activity_raw.shape[1]} columns")

## First look only

Show five source rows and inferred column types to become familiar with the file.
These rows are a preview, not a representative sample or a finding about players.
Dates remain as loaded until the later preparation stage.


In [ ]:
with pd.option_context("display.max_columns", None, "display.max_colwidth", 40):
    display(activity_raw.head(5))

display(activity_raw.dtypes.rename("Loaded dtype").to_frame())

## Analysis roadmap

1. **Validate the data contract.** Check player-day uniqueness, day-0 coverage,
   cohort dates, complete follow-up windows, nulls, date/tenure consistency,
   plan/status transitions, post-cancellation rows, nonnegative feature counts,
   and daily plan limits. Compare observed coverage with the source description.
2. **Prepare reproducibly.** Preserve `activity_raw` and source bytes. Parse dates
   into a separate table and produce a rule summary and row-level cleaning audit
   from the same code as any cleaning. If no cleaning is required, document the
   checks and that outcome. Reconcile rows and players before calculating metrics.
3. **Define the player cohort and metrics.** Build one row per day-0 player with
   observed initial plan. Count new accounts by that plan and cohort date.
   Define conversion as initially Free players with a subsequent Paid observation,
   divided by initially Free players; acknowledge day-0 same-day upgrades are unobservable.
   Count unique players with explicit Paid cancellation records. If reporting a
   cancellation proportion, define the denominator as players ever observed Paid,
   distinguish initially Paid from converters, and account for unequal Paid exposure.
4. **Calculate retention.** Use fixed day-0 cohort denominators, overall and by
   starting plan/cohort. Report exact-day activity separately from any-activity
   tenure weeks: week 1 = days 0–6, week 2 = 7–13, etc. Label days 56–59 as a
   partial final week. Verify follow-up availability before interpreting missing days.
5. **Explore journeys and associations after the mandatory metrics.** Examine
   upgrade timing, feature success/error rates (errors divided by attempts), and
   behavior segments. Define early exposure windows before later outcomes,
   address zero attempts and plan limits, and avoid using future activity to
   predict earlier outcomes. Repeated player-days are not independent observations.
   Predictive modeling is optional and only justified by a clear decision; if used,
   split by player before learned preprocessing and compare with a simple baseline.
6. **Verify and communicate.** Independently recalculate a headline count/rate
   from source-level records; reconcile denominators and boundary cases. Build
   checked charts and audience-specific recommendations with uncertainty and
   synthetic-data limitations. Create the requested deck only at that later stage.

**Current progress:** setup, audited preparation, mandatory questions and the
four additional goals are executed below. Early predictors and later outcomes
were confirmed with the user before fitting the association models.


## Cleaning contract and audit policy

We are preparing data for the four mandatory questions, without calculating their
aggregate answers. Keep every original column and add clearly named derived fields.
`source_row_id` is the CSV data-record number (1-based, excluding the header).

| Rule | Action and reason |
| --- | --- |
| Source lineage | Attach a stable source reference before preparation. |
| Contract checks | Validate keys, missingness, dates, day-0 coverage, transitions, cancellations and limits. Stop on conflicts rather than silently repair or exclude. |
| Dates and tenure | Parse explicit month/day/year dates into new fields; derive tenure and tenure week and compare supplied fields. |
| Activity flags and attempts | A source row is observed activity, even if all feature counts are zero. Explicit status alone identifies cancellation. Attempts = successes + errors. |
| Player roster | Anchor to day 0; derive first Paid and explicit cancellation dates, retaining nonconverters and noncancellers. |
| Calendar | Add days 0–59 per player; mark observed activity but leave absent plan/status/feature values missing. |
| Reconciliation | Check source values, joins, lineage and independent source-ledger calculations before export. |

`checks/transformation_summary.csv` records each applied rule and population
impact. `checks/cell_edits.csv` logs new field values and constructed calendar
rows, with keys and source references. Additions are distinguished from edits;
no original value is overwritten. Failed checks are saved before execution stops.


In [ ]:
output_folder = case_folder / "outputs" / "cleaning"
checks_folder = output_folder / "checks"
checks_folder.mkdir(parents=True, exist_ok=True)

# Work on a copy and retain a permanent reference to every CSV data record.
original_columns = activity_raw.columns.tolist()
activity_clean = activity_raw.copy(deep=True)
activity_clean.insert(0, "source_row_id", np.arange(1, len(activity_clean) + 1))
audit = AuditTrail(activity_clean, checks_folder)
audit.record_rule("source_lineage", "Trace every source record without modifying inputs",
    activity_raw, activity_clean, len(activity_clean), activity_clean.userid.nunique(),
    "Added 1-based CSV data-record ID; no filtering or source-value edits")
audit.audit_added_fields(activity_clean, ["source_row_id"], "source_lineage", "activity_clean", "date")

# Parse dates explicitly while preserving the original text fields.
activity_clean["activity_date"] = pd.to_datetime(activity_clean.date, format="%m/%d/%Y", errors="coerce")
activity_clean["cohort_date"] = pd.to_datetime(activity_clean.startdate, format="%m/%d/%Y", errors="coerce")
activity_clean["tenure_day"] = (activity_clean.activity_date - activity_clean.cohort_date).dt.days

### Check dates, players and billing history

In [ ]:
validation.check_dates_and_cohort(
    activity_clean=activity_clean,
    activity_raw=activity_raw,
    audit=audit,
)

feature_columns = validation.check_feature_limits(
    activity_clean=activity_clean,
    audit=audit,
    original_columns=original_columns,
)

cancellation_date_by_user = validation.check_billing_history(activity_clean=activity_clean, audit=audit)

### Review the saved quality checks

In [ ]:
quality_checks = audit_helpers.save_quality_checks(audit=audit, checks_folder=checks_folder)

display(quality_checks)

## Prepare the observed activity table

The contract checks above determine whether preparation can proceed. The next
steps add typed dates, verified tenure, week labels, observed-activity and explicit
cancellation flags, and attempt totals. All original source columns remain intact.
Week 1 covers days 0–6; week 9 covers days 56–59 and is explicitly partial.
Observed activity is defined by the presence of a source row, not feature usage.


### Add analysis fields with an audit trail

In [ ]:
activity_clean, date_fields, derived_fields = preparation.derive_activity_fields(
    activity_clean=activity_clean,
    activity_raw=activity_raw,
    audit=audit,
)

display(activity_clean.head(5))

## Prepare the player table

Anchor the roster on each player's validated day-0 record. Dates for first Paid
observation and explicit cancellation come from all that player's source records;
the audit lineage maps those records to each player. An absent event date means
**not observed within the window**, not that the event can never occur.

`converted_from_initial_free` applies only to initially Free players. Plan on day 0
is an end-of-day observation; same-day signup/upgrade ordering remains unobservable.
The follow-up cutoff is the source statement's May 5 end date, not last activity.
Full 60-day data availability relies on the source's complete synthetic-file
declaration; coverage checks cannot independently prove telemetry completeness.


### One record per player

In [ ]:
player_summary, player_fields = preparation.build_player_table(
    activity_clean=activity_clean,
    audit=audit,
    cancellation_date_by_user=cancellation_date_by_user,
    checks_folder=checks_folder,
)

display(player_summary[["userid", "cohort_date", "initial_plan", "window_end_date"]].head())

## Construct a complete retention calendar

Create 60 rows per day-0 player, including after explicit cancellation, to preserve
the original retention denominator. `observed_activity=False` means no source
activity row. It does not infer billing status or cancellation. For added rows,
plan, Status and feature counts stay missing; no forward filling is performed.
The row-level ledger logs every added calendar row, while `source_row_id` links
every observed row back to the file. This calendar is a new output table, not an
expansion of the cleaned observed-activity table.


### Build the complete retention calendar

In [ ]:
player_day_calendar, added_calendar_rows = preparation.build_retention_calendar(
    activity_clean=activity_clean,
    audit=audit,
    feature_columns=feature_columns,
    player_summary=player_summary,
)

In [ ]:
week_windows = pd.DataFrame({"tenure_week": np.arange(1, 10)})
week_windows["start_day_inclusive"] = (week_windows.tenure_week - 1) * 7
week_windows["end_day_inclusive"] = (week_windows.start_day_inclusive + 6).clip(upper=59)
week_windows["days_in_window"] = week_windows.end_day_inclusive - week_windows.start_day_inclusive + 1
week_windows["is_partial_week"] = week_windows.days_in_window.lt(7)
display(week_windows)
print(f"Prepared {len(player_day_calendar):,} player-days; {len(added_calendar_rows):,} constructed rows clearly labeled.")

## Reconcile and independently verify the preparation

Check source preservation and audit completeness, then independently read the CSV
using Python's standard-library reader. That separate path reconstructs each
player's day-0 plan, first Paid date, cancellation date, and activity keys, and
compares them with the prepared outputs. This verifies preparation without
reporting the four business answers. Assertions stop exports if a check fails.


### Reconcile source values and retention denominators

In [ ]:
validation.verify_preservation_and_calendar(
    activity_clean=activity_clean,
    activity_raw=activity_raw,
    added_calendar_rows=added_calendar_rows,
    audit=audit,
    data_path=data_path,
    feature_columns=feature_columns,
    original_columns=original_columns,
    player_day_calendar=player_day_calendar,
    player_summary=player_summary,
    source_sha256=source_sha256,
)

### Independently check each player's event dates

In [ ]:
source_ledger, source_keys = validation.read_player_ledger(data_path=data_path)

validation.verify_player_events(
    audit=audit,
    player_day_calendar=player_day_calendar,
    player_summary=player_summary,
    source_keys=source_keys,
    source_ledger=source_ledger,
)

### Reconcile the additions log

In [ ]:
cell_edits, transformation_summary = audit_helpers.reconcile_added_fields(
    activity_clean=activity_clean,
    added_calendar_rows=added_calendar_rows,
    audit=audit,
    checks_folder=checks_folder,
    date_fields=date_fields,
    derived_fields=derived_fields,
    player_fields=player_fields,
    player_summary=player_summary,
)

display(transformation_summary)
display(pd.DataFrame(audit.validation_results))

## Save prepared data and the audit trail

Parquet preserves dates, booleans and nullable integers. The CSV audit files remain
readable without Python. Reload checks verify that the saved tables match the
in-memory data. The player and calendar outputs are ready for the four mandatory
questions, subject to the source's coverage declaration and day-0 plan limitation.
No aggregate retention, conversion, plan split, or cancellation results are produced.


### Save the prepared data

In [ ]:
preparation.save_prepared_tables(
    activity_clean=activity_clean,
    audit=audit,
    output_folder=output_folder,
    player_day_calendar=player_day_calendar,
    player_summary=player_summary,
    week_windows=week_windows,
)

### Record dimensions, checks and reproduction details

In [ ]:
manifest = reporting.save_preparation_manifest(
    activity_clean=activity_clean,
    activity_raw=activity_raw,
    added_calendar_rows=added_calendar_rows,
    audit=audit,
    cell_edits=cell_edits,
    checks_folder=checks_folder,
    data_path=data_path,
    environment=environment,
    output_folder=output_folder,
    player_day_calendar=player_day_calendar,
    player_summary=player_summary,
    project_root=project_root,
    quality_checks=quality_checks,
    source_sha256=source_sha256,
)

reporting.write_preparation_record(
    case_folder=case_folder,
    manifest=manifest,
    source_sha256=source_sha256,
)

## Preparation outcome and handoff

The executed checks and manifest above reconcile the complete input population
to the prepared tables and audit trail. No defects required source-value edits,
exclusions or imputation. The additions are typed/derived fields, one-row-per-player
summaries, and an explicitly labeled retention calendar.

| Mandatory question | Ready-to-use table and fields | Denominator or boundary |
| --- | --- | --- |
| New Free/Paid accounts | `player_summary`: `userid`, `cohort_date`, `initial_plan` | One validated day-0 player; observed end-of-day plan |
| Daily/weekly retention | `player_day_calendar`: `userid`, `tenure_day`, `tenure_week`, `observed_activity`, `is_partial_week` | Original day-0 cohort; week 9 is partial; source-declared full coverage |
| Free-to-Paid conversion | `player_summary`: `initially_free`, `converted_from_initial_free`, `conversion_date`, `conversion_tenure_day` | All initially Free players, including nonconverters |
| Paid cancellations | `player_summary`: `ever_observed_paid`, `explicit_cancellation_observed`, `cancellation_date`, `initial_plan` | Count distinct explicit cancellation events; any rate requires an explicit Paid denominator |

Reload with `pd.read_parquet(output_folder / "player_summary.parquet")` (and
similarly for `activity_clean` or `player_day_calendar`). Parquet retains field
types. Original field definitions remain in `source/README.md`; new-field
definitions and output paths are in the case `README.md`.

**Stopping point:** preparation is complete. Aggregate new-account counts by
plan, retention, conversion, cancellation answers, charts and modeling remain pending.


# Mandatory questions

**Readiness: proceed with documented limitations.** The 29 data-contract and 20
preparation-reconciliation checks above passed. The calculations below reload
the saved cleaned tables. We describe the entire supplied synthetic cohort;
these are observed counts and proportions, not population forecasts or causal
effects, so sampling confidence intervals are not added.

Full follow-up relies on the pinned source's complete-data declaration. Starting
plan is observed at day-end on day 0; same-day signup/upgrade ordering cannot be
recovered. Every question has its own code cell, followed by a verification cell.


## Question 1 — How many new Free and Paid accounts were created March 3–7?

Use `player_summary`: one row per day-0 player. Count by **initial plan**, rather
than current/last plan, so later upgrades do not change the starting-plan counts.
Show the full enrollment window and each start date as a reconciliation.


In [ ]:
# Reload the saved player table so these answers use the cleaned outputs.
players = pd.read_parquet(output_folder / "player_summary.parquet")
assert players.userid.is_unique, "Player summary must contain exactly one row per player."
assert players.cohort_date.between("2025-03-03", "2025-03-07").all()
assert players.full_60_day_window_available.all(), "Review incomplete follow-up before answering."

results_folder = case_folder / "outputs" / "mandatory_questions"
tables_folder = results_folder / "tables"
tables_folder.mkdir(parents=True, exist_ok=True)

new_accounts = players.groupby("initial_plan").size().reindex(["Free", "Paid"], fill_value=0)
new_accounts = new_accounts.rename("new_players").rename_axis("initial_plan").reset_index()
new_accounts_by_date = pd.crosstab(players.cohort_date, players.initial_plan)
new_accounts_by_date = new_accounts_by_date.reindex(columns=["Free", "Paid"], fill_value=0)
new_accounts_by_date["Total"] = new_accounts_by_date.sum(axis=1)
assert new_accounts.new_players.sum() == len(players), "Starting-plan counts must cover the entire cohort."

display(new_accounts)
display(new_accounts_by_date)
initial_counts = new_accounts.set_index("initial_plan").new_players
print(f"Answer: {initial_counts['Free']:,} initially Free and {initial_counts['Paid']:,} initially Paid players; {len(players):,} total.")
print("These are observed day-0 end-of-day plans, not separately logged signup-plan events.")
new_accounts.to_csv(tables_folder / "q1_new_accounts.csv", index=False)
new_accounts_by_date.to_csv(tables_folder / "q1_new_accounts_by_date.csv")

## Question 2 — How many players return in subsequent days and weeks? What is retention?

Use `player_day_calendar`. **Daily retention** = distinct players active on that
exact tenure day / all day-0 players. **Weekly retention** = distinct players
active at least once in that tenure-week window / all day-0 players. A player
active on several days counts once in their week's numerator.

The denominator stays fixed, including after cancellation or nonreturn. These
are tenure days/weeks since each player's start date, not calendar weeks.
Week 1 (days 0–6) includes signup and is 100% by construction; week 9 (days 56–59)
is partial. Missing source rows mean no observed activity, not cancellation.


In [ ]:
calendar = pd.read_parquet(output_folder / "player_day_calendar.parquet")
assert not calendar.duplicated(["userid", "tenure_day"]).any()
assert set(calendar.userid) == set(players.userid)
assert calendar.groupby("userid").size().eq(60).all()
assert calendar.full_60_day_window_available.all()
cohort_players = len(players)

daily_retention, weekly_retention = metrics.retention_tables(
    calendar=calendar,
    cohort_players=cohort_players,
    output_folder=output_folder,
)

print("Daily retention — active on that exact tenure day:")
with pd.option_context("display.max_rows", 60):
    display(daily_retention.style.format({"retention_pct": "{:.1f}%"}))
print("Weekly retention — active at least once in the indicated tenure-day window:")
display(weekly_retention.style.format({"retention_pct": "{:.1f}%"}))
for day in [1, 7, 14, 30, 59]:
    result = daily_retention.loc[daily_retention.tenure_day.eq(day)].iloc[0]
    print(f"Day {day}: {int(result.active_players):,}/{cohort_players:,} active ({result.retention_pct:.1f}%).")
print("Day 7 retention is exact-day activity; week 2 retention covers any activity on days 7–13.")
print("Week 1 includes day 0. The partial final week has fewer chances to observe a return than a full week.")
daily_retention.to_csv(tables_folder / "q2_daily_retention.csv", index=False)
weekly_retention.to_csv(tables_folder / "q2_weekly_retention.csv", index=False)

## Question 3 — How many Free players upgraded to Paid? What is conversion?

Use `player_summary`. The denominator is **all initially Free players**, including
those who never return. The numerator is initially Free players subsequently
observed Paid within days 0–59. Initially Paid players are excluded from both.
This measures observed conversion from the day-0 Free state; same-day signup
and upgrade events before that snapshot are not recoverable.


In [ ]:
initially_free_players = players.loc[players.initially_free].copy()
free_players_count = len(initially_free_players)
converted_players_count = int(initially_free_players.converted_from_initial_free.sum())
conversion_pct = 100 * converted_players_count / free_players_count if free_players_count else np.nan

conversion_summary = pd.DataFrame([{
    "initially_free_players": free_players_count,
    "converted_to_paid_players": converted_players_count,
    "not_observed_converted_players": free_players_count - converted_players_count,
    "conversion_pct": conversion_pct,
    "tenure_window": "days 0–59; conversion after observed day-0 Free state",
}])
display(conversion_summary.style.format({"conversion_pct": "{:.1f}%"}))
print(f"Answer: {converted_players_count:,} of {free_players_count:,} initially Free players upgraded; conversion = {conversion_pct:.1f}%.")
print("The remaining players were not observed converting within this window; this does not mean they can never convert.")
conversion_summary.to_csv(tables_folder / "q3_free_to_paid_conversion.csv", index=False)

## Question 4 — How many Paid players cancelled?

Use `player_summary` and count players with **explicit cancellation records**.
Include both initially Paid players and initially Free players who converted.
Show those groups separately to make the count auditable. The question asks for
a count; these groups have different Paid exposure lengths, so their counts
should not be interpreted as comparable cancellation risks or monthly churn rates.


In [ ]:
paid_players = players.loc[players.ever_observed_paid].copy()
assert not players.loc[players.explicit_cancellation_observed, "ever_observed_paid"].eq(False).any()
cancelled_players_count = int(paid_players.explicit_cancellation_observed.sum())

cancellation_summary = pd.DataFrame([{
    "ever_observed_paid_players": len(paid_players),
    "explicitly_cancelled_players": cancelled_players_count,
    "no_recorded_cancellation_players": len(paid_players) - cancelled_players_count,
}])
cancellations_by_starting_plan = paid_players.groupby("initial_plan", as_index=False).agg(
    ever_observed_paid_players=("userid", "size"),
    explicitly_cancelled_players=("explicit_cancellation_observed", "sum"),
)
assert cancellations_by_starting_plan.explicitly_cancelled_players.sum() == cancelled_players_count
display(cancellation_summary)
display(cancellations_by_starting_plan)
print(f"Answer: {cancelled_players_count:,} distinct Paid players have explicit cancellation records within days 0–59.")
print("Paid players include both day-0 Paid accounts and subsequent Free-to-Paid converters.")
print("No recorded cancellation is not proof of current subscription status after the file ends; missing activity alone is not cancellation.")
cancellation_summary.to_csv(tables_folder / "q4_paid_cancellations.csv", index=False)
cancellations_by_starting_plan.to_csv(tables_folder / "q4_cancellations_by_starting_plan.csv", index=False)

## Independent verification and saved answer summary

Reread the original CSV with `csv.DictReader`. Independently count day-0 plan
memberships, sets of active players per day/week, initially Free players later
observed Paid, and explicitly cancelled players. Compare all answers and every
retention numerator/percentage with the cleaned-table route. No new cleaning
is applied. Recheck that the source fingerprint is unchanged.


### Check all four answers through an independent CSV calculation

In [ ]:
answer_checks = validation.verify_mandatory_answers(
    cancelled_players_count=cancelled_players_count,
    conversion_pct=conversion_pct,
    converted_players_count=converted_players_count,
    daily_retention=daily_retention,
    data_path=data_path,
    free_players_count=free_players_count,
    initial_counts=initial_counts,
    paid_players=paid_players,
    results_folder=results_folder,
    source_sha256=source_sha256,
    weekly_retention=weekly_retention,
)

display(pd.DataFrame([{"check": name, "passed": passed} for name, passed in answer_checks.items()]))

### Save the mandatory-answer summary

In [ ]:
answer_brief = reporting.write_mandatory_answers(
    answer_checks=answer_checks,
    cancelled_players_count=cancelled_players_count,
    case_folder=case_folder,
    cohort_players=cohort_players,
    conversion_pct=conversion_pct,
    converted_players_count=converted_players_count,
    daily_retention=daily_retention,
    free_players_count=free_players_count,
    initial_counts=initial_counts,
    paid_players=paid_players,
    results_folder=results_folder,
    weekly_retention=weekly_retention,
)

display(Markdown(answer_brief))

# Additional goals: feature associations, segments and conversion journeys

**User-confirmed definitions:** predictors are measured during days 0–6.
Retention means any observed activity during days 28–34. Conversion means a
first Paid observation during days 7–59 among initially Free players not yet
converted by day 6. One player is one independent observation for these models.

The plan below is fixed before inspecting feature/outcome associations.
These are exploratory associations in synthetic data, not causal effects or
validated forecasts. No feature or segment is selected because of its outcome.

1. Report rank correlations, with permutation p-values and multiplicity adjustment.
2. Fit separate retention and conversion logistic regressions using all three
   first-week feature volumes, active days, and initial Paid status for retention.
3. Fit one error-rate model per feature/outcome, controlling for that feature's
   volume, active days, and initial Paid status where relevant. Rates remain
   undefined when attempts are zero; exclusions are reported rather than imputed.
4. Compare prespecified weekend/weekday and plan-limit-normalized feature
   preference groups, then fit adjusted group models.
5. Report conversion risk sets, event timing, pre-upgrade journeys and a discrete
   daily conversion-hazard regression with player-clustered uncertainty.

Intervals are pointwise model-based 95% intervals. Benjamini–Hochberg q-values
summarize exploratory multiple comparisons, separately for correlations and
the 12 primary feature slopes. The 33 conversions limit precision and model
complexity. We also check simpler specifications and first-week plan differences.

In [ ]:
association_folder = case_folder / "outputs" / "additional_analysis"
association_tables = association_folder / "tables"
association_checks = association_folder / "checks"
association_figures = association_folder / "figures"
for folder in [association_tables, association_checks, association_figures]:
    folder.mkdir(parents=True, exist_ok=True)

analysis_plan = reporting.save_association_plan(association_checks=association_checks)

activity = pd.read_parquet(output_folder / "activity_clean.parquet")
roster = pd.read_parquet(output_folder / "player_summary.parquet")
calendar = pd.read_parquet(output_folder / "player_day_calendar.parquet")

### Aggregate first-week predictors and define later outcomes

In [ ]:
early_activity, early_features, feature_map = features.first_week_features(activity=activity)

model_players, conversion_players, samples, sample_summary = features.define_later_outcomes(
    association_tables=association_tables,
    calendar=calendar,
    early_features=early_features,
    roster=roster,
)

display(sample_summary)

### Save feature lineage and model eligibility

In [ ]:
feature_audit = audit_helpers.save_first_week_lineage(
    association_checks=association_checks,
    calendar=calendar,
    conversion_players=conversion_players,
    early_activity=early_activity,
    early_features=early_features,
    feature_map=feature_map,
    model_players=model_players,
)

## Feature profile and correlation analysis

Totals refer to the full seven-calendar-day window. A player with fewer active
days has fewer observed attempts, which motivates adjustment for active days.
Zero attempts give a genuine zero volume but an undefined error rate.
Rank correlation with a binary outcome compares predictor ranks in outcome groups.
Permuting outcome labels gives an exploratory unadjusted test of independence.
It does not adjust for starting plan, general engagement, or other features.

In [ ]:
profile_fields = ["active_days"] + [f"{feature}_{suffix}" for feature in feature_map for suffix in ["attempts", "errors", "error_rate"]]
feature_profile = model_players[profile_fields].describe().T
feature_profile["missing_players"] = model_players[profile_fields].isna().sum()
feature_profile.to_csv(association_tables / "first_week_feature_profile.csv")
display(feature_profile)

In [ ]:
correlations = features.correlate_features_and_outcomes(
    association_tables=association_tables,
    feature_map=feature_map,
    samples=samples,
)

display(correlations.round(4))

In [ ]:
plotting.correlation_heatmap(
    association_figures=association_figures,
    association_tables=association_tables,
    correlations=correlations,
)

## Logistic regression method and diagnostics

Logistic regression models the log odds of a binary outcome. We report odds
ratios (ORs) and 95% intervals, not percentage-point effects. A volume coefficient
is per doubling of **1 + first-week attempts**. Error-rate coefficients are per
10 percentage points. OR > 1 means a positive association, conditional on controls.

Primary usage formulas are visible in the next cell. HC1 standard errors treat
players as independent. Error models include one feature at a time to keep
complexity manageable and explicitly control its attempt volume. These models
do not mutually adjust all three error rates. Only 33 conversion events means
wide intervals and unstable estimates are possible. No held-out prediction
claims are made. We check convergence, warnings, rank, condition number and VIF.
See [statsmodels GLM documentation](https://www.statsmodels.org/stable/generated/statsmodels.genmod.generalized_linear_model.GLM.html).

In [ ]:
from statsmodels.stats.multitest import multipletests

# The collection records each fit and its diagnostics; formulas remain visible.
models = RegressionCollection()
usage_terms = "log2_optimizer + log2_coach + log2_agent + active_days"
retention_formula = "retained ~ " + usage_terms + " + initial_paid"
conversion_formula = "later_conversion ~ " + usage_terms
print("Retention model:", retention_formula)
print("Conversion model:", conversion_formula)

## Goal 1 — Feature usage versus retention and conversion

The retention model uses all 300 players and controls initial plan. The
conversion model uses only eligible initially Free players. Active days is
included to distinguish feature volume from general first-week engagement.
All three feature volumes are included together. Adjustment describes
associations conditional on these variables and does not identify causal effects.

In [ ]:
retention_usage = models.fit("retention_usage", model_players, retention_formula)
conversion_usage = models.fit("conversion_usage", conversion_players, conversion_formula)
primary_usage = pd.concat([retention_usage, conversion_usage], ignore_index=True)
primary_usage = primary_usage.loc[primary_usage.term.isin(["log2_optimizer", "log2_coach", "log2_agent"])].copy()
display(primary_usage[["model", "term", "odds_ratio", "ci_low", "ci_high", "p_value"]].round(4))
primary_usage.to_csv(association_tables / "usage_regression_coefficients.csv", index=False)

### Compare simpler usage specifications

In [ ]:
modeling.fit_usage_sensitivities(
    feature_map=feature_map,
    models=models,
    samples=samples,
    usage_terms=usage_terms,
)

## Goal 2 — Feature errors versus retention and conversion

For each feature, error rate = first-week errors / first-week attempts.
Players with zero attempts are excluded only from that feature's error model;
zero errors among positive attempts remain valid zero rates. The model controls
that feature's log attempt volume and active days, plus initial plan for retention.
Avoid reading raw error-count correlations as the consequence of errors.

In [ ]:
primary_errors, error_samples = modeling.fit_error_models(feature_map=feature_map, models=models, samples=samples)

display(pd.DataFrame(error_samples))
display(primary_errors[["model", "term", "odds_ratio", "ci_low", "ci_high", "p_value"]].round(4))
pd.DataFrame(error_samples).to_csv(association_tables / "error_model_samples.csv", index=False)
primary_errors.to_csv(association_tables / "error_regression_coefficients.csv", index=False)
primary_associations = pd.concat([primary_usage, primary_errors], ignore_index=True)
primary_associations["bh_q"] = multipletests(primary_associations.p_value, method="fdr_bh")[1]
primary_associations.to_csv(association_tables / "primary_associations.csv", index=False)
display(primary_associations[["model", "term", "odds_ratio", "ci_low", "ci_high", "p_value", "bh_q"]].round(4))

In [ ]:
plotting.feature_association_intervals(
    association_figures=association_figures,
    primary_associations=primary_associations,
)

## Goal 3 — Differences between player groups

Segments use only days 0–6 and are fixed before looking at later outcomes.

**Weekend preference:** compare active weekend days / 2 with active weekdays / 5.
Each complete seven-day tenure window contains two weekend and five weekday
opportunities. Strictly larger activity frequency determines preference; ties
are a separate balanced group. This is activity preference, not acquisition day.

**Feature preference:** sum daily attempts divided by that day's plan limit,
then compare the three features. This corrects for very different usage caps.
The largest average fraction of daily allowance determines the dominant feature;
exact ties have their own label. Dominance measures relative preference, not
an absolute heavy-user threshold. A player can have low overall usage.

Group tables include player counts, event counts, rates and Wilson intervals.
Adjusted segment regressions include active days and initial plan for retention.
Small groups or model warnings limit interpretation. These comparisons are
exploratory and not independent of the earlier feature comparisons.
A revision after the initial diagnostic failure restricts adjusted group models
to categories with at least 10 players. Sparse groups remain in the tables;
this restriction is based on group size, not the direction of the association.

### Assign groups using only first-week behavior

In [ ]:
model_players, conversion_players, normalized_fields = features.assign_player_segments(
    early_activity=early_activity,
    feature_map=feature_map,
    model_players=model_players,
)

### Compare all groups and fit estimable contrasts

In [ ]:
segment_summary, segment_model_scope = modeling.fit_segment_comparisons(
    conversion_players=conversion_players,
    model_players=model_players,
    models=models,
)

segment_coefficients = reporting.save_segment_comparisons(
    association_checks=association_checks,
    association_tables=association_tables,
    model_players=model_players,
    models=models,
    normalized_fields=normalized_fields,
    segment_model_scope=segment_model_scope,
    segment_summary=segment_summary,
)

display(segment_summary.round(2))
display(pd.DataFrame(segment_model_scope))
display(segment_coefficients[["model", "term", "odds_ratio", "ci_low", "ci_high", "bh_q"]].round(4))

## Goal 4 — Conversion timing and player journeys

First, describe all initially Free players' conversion timing from day 1 onward.
A daily risk set includes everyone not previously converted, even if inactive.
There are no Free cancellations in the source contract. A window's conditional
conversion percentage uses Free players at its start, not just eventual converters.
Sparse daily peaks are unstable, so show grouped windows as well.

For a regression consistent with the agreed landmark, use at-risk player-days
from days 7–59. Outcome = conversion that day. First-week usage predictors are
fixed at day 6, never updated with post-upgrade activity. Include log tenure day
as a parsimonious baseline timing adjustment and cluster uncertainty by player.
This assumes a smooth log-tenure pattern; the empirical hazard table remains
the direct timing answer. Repeated days do not create more than 33 conversion
events. Results describe observed conversion and do not measure intervention lift.

Journeys show feature attempts on observed days before the upgrade, alongside
the proportion active on each relative day. The upgrade day is excluded because
plan and feature counts share an end-of-day snapshot. This converter-only view
is descriptive and cannot show that a change in usage caused an upgrade.

### Observe conversion timing using Free players still at risk

In [ ]:
initial_free_roster, conversion_hazard, conversion_windows = metrics.conversion_risk_sets(
    association_tables=association_tables,
    model_players=model_players,
)

display(conversion_windows.round(2))

### Prepare daily risk rows

In [ ]:
risk_days = features.at_risk_player_days(conversion_players=conversion_players)

In [ ]:
hazard_formula = "converted_today ~ log_tenure_day + log2_optimizer + log2_coach + log2_agent + active_days"
hazard_coefficients = models.fit("daily_conversion_hazard", risk_days, hazard_formula, cluster_column="userid")
hazard_coefficients.to_csv(association_tables / "conversion_hazard_regression.csv", index=False)
display(hazard_coefficients[["term", "odds_ratio", "ci_low", "ci_high", "p_value"]].round(4))
print("Hazard ORs refer to conditional daily conversion odds, not 53-day cumulative conversion probabilities.")

### Summarize the week before conversion

In [ ]:
converters, journey_summary = metrics.preconversion_journeys(
    activity=activity,
    association_tables=association_tables,
    calendar=calendar,
    initial_free_roster=initial_free_roster,
)

display(journey_summary.round(2))

In [ ]:
plotting.conversion_timing_and_journey(
    association_figures=association_figures,
    conversion_windows=conversion_windows,
    journey_summary=journey_summary,
)

## Diagnostics, sensitivity and independent verification

Compare the same feature's joint usage model with its simpler feature-specific
model and restrict retention to initially Free players. For the daily hazard,
add quadratic log tenure as a prespecified functional-form sensitivity.
Inference remains exploratory. The independent route below rebuilds all
first-week totals, retention outcomes, and later-conversion counts from the CSV,
checks rank correlations, and refits the two main usage models with SciPy's
Bernoulli likelihood rather than statsmodels' fitting routine.

In [ ]:
risk_days["log_tenure_centered"] = risk_days.log_tenure_day - risk_days.log_tenure_day.mean()
risk_days["log_tenure_squared"] = risk_days.log_tenure_centered ** 2
models.fit("daily_conversion_hazard_quadratic_time", risk_days,
    hazard_formula.replace("log_tenure_day", "log_tenure_centered") + " + log_tenure_squared", cluster_column="userid")

### Review diagnostics and specification sensitivity

In [ ]:
all_coefficients, diagnostics = reporting.export_model_diagnostics(
    association_checks=association_checks,
    association_tables=association_tables,
    models=models,
)

sensitivity_summary, hazard_sensitivity = reporting.export_sensitivities(
    all_coefficients=all_coefficients,
    association_tables=association_tables,
    feature_map=feature_map,
)

display(diagnostics[["model", "players", "events", "parameters", "converged", "max_vif", "warnings"]].round(2))
display(sensitivity_summary.round(3))

### Independently verify features, outcomes and conversion risk sets

In [ ]:
source_later_converters = validation.verify_features_and_risk_sets(
    conversion_hazard=conversion_hazard,
    conversion_players=conversion_players,
    data_path=data_path,
    feature_map=feature_map,
    model_players=model_players,
)

validation.verify_rank_correlations(correlations=correlations, samples=samples)

### Independently refit the main likelihoods

In [ ]:
validation.verify_regression_optimizers(association_checks=association_checks, models=models)

validation.verify_final_analysis_population(
    association_folder=association_folder,
    conversion_windows=conversion_windows,
    converters=converters,
    data_path=data_path,
    early_features=early_features,
    feature_audit=feature_audit,
    model_players=model_players,
    risk_days=risk_days,
    source_later_converters=source_later_converters,
    source_sha256=source_sha256,
)

## Findings and implications

The following summary is generated from the executed result tables. Raw
correlations and adjusted models answer different questions. Highlighted
associations are hypotheses for further research, not product-effect estimates.

### Read and save the checked findings

In [ ]:
findings = reporting.write_association_findings(
    association_folder=association_folder,
    conversion_players=conversion_players,
    conversion_windows=conversion_windows,
    converters=converters,
    correlations=correlations,
    model_players=model_players,
    primary_associations=primary_associations,
    segment_summary=segment_summary,
)

reporting.save_association_record(
    analysis_plan=analysis_plan,
    association_checks=association_checks,
    case_folder=case_folder,
    conversion_players=conversion_players,
    diagnostics=diagnostics,
    model_players=model_players,
    models=models,
    source_sha256=source_sha256,
)

display(Markdown(findings))